<a id="notebook-top"></a>
# Packaging, Deployment, Documentation, and Final Solution Readiness

**Notebook purpose.** This notebook packages the completed application for reproducible execution and assembles the final technical documentation. It validates the repository, launch contracts, container definitions, service connectivity, health checks, evidence, artifact checksums, and final solution readiness.

**What this notebook covers.** Runtime requirements; independent FastAPI and Streamlit launch contracts; Docker and Compose packaging; persistent volumes and health checks; README, architecture, usage, API, safety, testing, and MLflow documentation; demonstration evidence; registries; and final audits.

**Expected outcome.** A deployment and documentation package with checksum-validated artifacts, completed MLflow lineage, reproducibility evidence, and a final readiness gate for submission and demonstration.

**Safety boundary.** This work is intended for research and educational demonstration. Model outputs are not clinical diagnoses and must not replace qualified professional review.


<a id="notebook-index"></a>
## Notebook Index

Use the links below to move directly to a section. Each major section ends with a **Back to notebook index** link.

- [1. Packaging and Documentation Configuration](#nb09-1-packaging-and-documentation-configuration)
- [2. Final Repository Structure Audit](#nb09-2-final-repository-structure-audit)
- [3. Runtime Requirements](#nb09-3-runtime-requirements)
- [4. Environment and Independent Launch Contracts](#nb09-4-environment-and-independent-launch-contracts)
- [5. FastAPI Container Packaging](#nb09-5-fastapi-container-packaging)
- [6. Streamlit Container Packaging](#nb09-6-streamlit-container-packaging)
- [7. Compose Connectivity and Persistent-Volume Configuration](#nb09-7-compose-connectivity-and-persistent-volume-configuration)
- [8. Deployment Health Validation Contract](#nb09-8-deployment-health-validation-contract)
- [9. Solution README](#nb09-9-solution-readme)
- [10. Architecture Documentation](#nb09-10-architecture-documentation)
- [11. Usage and Deployment Instructions](#nb09-11-usage-and-deployment-instructions)
- [12. API Examples](#nb09-12-api-examples)
- [13. Safety and Limitations Documentation](#nb09-13-safety-and-limitations-documentation)
- [14. Testing, MLflow, and Reproducibility Instructions](#nb09-14-testing-mlflow-and-reproducibility-instructions)
- [15. Static Packaging Validation](#nb09-15-static-packaging-validation)
- [16. Demonstration Evidence Audit](#nb09-16-demonstration-evidence-audit)
- [17. Final Artifact Registry](#nb09-17-final-artifact-registry)
- [18. MLflow Packaging and Documentation Registration](#nb09-18-mlflow-packaging-and-documentation-registration)
- [19. Final Reproducibility and Storage Audit](#nb09-19-final-reproducibility-and-storage-audit)
- [20. Final Solution Readiness Gate](#nb09-20-final-solution-readiness-gate)


<a id="nb09-1-packaging-and-documentation-configuration"></a>
## 1. Packaging and Documentation Configuration

This section restores only the persisted paths and readiness evidence required for final packaging. It confirms that Notebook 8 completed successfully, prepares documentation and deployment directories, establishes the MLflow tracking URI, checks required tooling, and preserves the protected data-volume storage reserve without starting services or loading models.


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import shutil
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import yaml


SOLUTION_ROOT = Path("/home/jovyan/chest-xray-ai-assistant")
DATA_ROOT = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)
PATH_REGISTRY_PATH = SOLUTION_ROOT / "configs" / "paths.yaml"
CONFIG_ROOT = SOLUTION_ROOT / "configs"
API_ROOT = SOLUTION_ROOT / "api"
UI_ROOT = SOLUTION_ROOT / "ui"
SOURCE_ROOT = SOLUTION_ROOT / "src"
TEST_ROOT = SOLUTION_ROOT / "tests"
NOTEBOOK_ROOT = SOLUTION_ROOT / "notebooks"
REPORTS_ROOT = SOLUTION_ROOT / "reports"
DOCS_ROOT = SOLUTION_ROOT / "docs"
DEPLOYMENT_ROOT = SOLUTION_ROOT / "deployment"
OUTPUT_ROOT = DATA_ROOT / "outputs"
API_OUTPUT_ROOT = OUTPUT_ROOT / "api"
UI_OUTPUT_ROOT = OUTPUT_ROOT / "ui"
PACKAGING_OUTPUT_ROOT = OUTPUT_ROOT / "packaging"

NOTEBOOK8_READINESS_PATH = (
    UI_OUTPUT_ROOT / "streamlit_interface_integration_readiness.json"
)
EXPECTED_NOTEBOOK8_READINESS_VERSION = (
    "streamlit-interface-integration-readiness-v1"
)
EXPECTED_NOTEBOOK8_STATUS = "ready_for_packaging_and_deployment"
MINIMUM_FREE_STORAGE_GIB = 4.0
MLFLOW_TRACKING_URI = f"file://{DATA_ROOT / 'mlflow'}"

for required_directory in (
    SOLUTION_ROOT,
    DATA_ROOT,
    CONFIG_ROOT,
    API_ROOT,
    UI_ROOT,
    SOURCE_ROOT,
    TEST_ROOT,
    REPORTS_ROOT,
    OUTPUT_ROOT,
    API_OUTPUT_ROOT,
    UI_OUTPUT_ROOT,
):
    if not required_directory.is_dir():
        raise FileNotFoundError(
            f"Required persisted directory is unavailable: {required_directory}"
        )

if not PATH_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Path registry is unavailable: {PATH_REGISTRY_PATH}"
    )

if not NOTEBOOK8_READINESS_PATH.is_file():
    raise FileNotFoundError(
        "Notebook 8 readiness evidence is unavailable: "
        f"{NOTEBOOK8_READINESS_PATH}"
    )

path_registry = yaml.safe_load(
    PATH_REGISTRY_PATH.read_text(encoding="utf-8")
) or {}
notebook8_readiness = json.loads(
    NOTEBOOK8_READINESS_PATH.read_text(encoding="utf-8")
)

NOTEBOOK8_READINESS_CONFIRMED = (
    notebook8_readiness.get("readiness_version")
    == EXPECTED_NOTEBOOK8_READINESS_VERSION
    and notebook8_readiness.get("status") == EXPECTED_NOTEBOOK8_STATUS
    and isinstance(notebook8_readiness.get("checks"), dict)
    and notebook8_readiness["checks"]
    and all(value is True for value in notebook8_readiness["checks"].values())
)

if not NOTEBOOK8_READINESS_CONFIRMED:
    raise RuntimeError(
        "Notebook 8 readiness evidence does not confirm successful "
        "Streamlit interface integration."
    )

DOCS_ROOT.mkdir(parents=True, exist_ok=True)
DEPLOYMENT_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.update(
    {
        "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
        "HF_HOME": str(DATA_ROOT / "hf-cache"),
        "HF_DATASETS_CACHE": str(DATA_ROOT / "hf-cache" / "datasets"),
        "TORCH_HOME": str(DATA_ROOT / "models" / "torch-cache"),
        "TOKENIZERS_PARALLELISM": "false",
    }
)

required_packages = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "streamlit": "streamlit",
    "httpx": "httpx",
    "PyYAML": "yaml",
    "mlflow": "mlflow",
}
package_versions = {}

for distribution_name, module_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        raise ModuleNotFoundError(
            f"Required packaging dependency is unavailable: {distribution_name}"
        )
    try:
        package_versions[distribution_name] = version(distribution_name)
    except PackageNotFoundError:
        package_versions[distribution_name] = "AVAILABLE"

free_storage_gib = shutil.disk_usage(DATA_ROOT).free / (1024 ** 3)
STORAGE_RESERVE_PRESERVED = free_storage_gib >= MINIMUM_FREE_STORAGE_GIB

if not STORAGE_RESERVE_PRESERVED:
    raise RuntimeError(
        f"Protected storage reserve unavailable: {free_storage_gib:.2f} GiB free."
    )

print("PACKAGING AND DOCUMENTATION CONFIGURATION")
print("-" * 110)
print(f"Solution root               : {SOLUTION_ROOT}")
print(f"Data root                   : {DATA_ROOT}")
print(f"Notebook 8 readiness        : PASS")
print(f"Notebook 8 status           : {notebook8_readiness['status']}")
print(f"Deployment root             : {DEPLOYMENT_ROOT}")
print(f"Documentation root          : {DOCS_ROOT}")
print(f"Packaging output root       : {PACKAGING_OUTPUT_ROOT}")
print(f"MLflow tracking URI         : {MLFLOW_TRACKING_URI}")
print(f"Required packages           : {len(package_versions)} / {len(required_packages)}")
print(f"Free data-volume storage    : {free_storage_gib:.2f} GiB")
print(f"Protected storage reserve   : PASS")
print(f"Models loaded               : NO")
print(f"Models retrained            : NO")
print()
print("READY FOR FINAL REPOSITORY STRUCTURE AUDIT")


PACKAGING AND DOCUMENTATION CONFIGURATION
--------------------------------------------------------------------------------------------------------------
Solution root               : /home/jovyan/chest-xray-ai-assistant
Data root                   : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
Notebook 8 readiness        : PASS
Notebook 8 status           : ready_for_packaging_and_deployment
Deployment root             : /home/jovyan/chest-xray-ai-assistant/deployment
Documentation root          : /home/jovyan/chest-xray-ai-assistant/docs
Packaging output root       : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/packaging
MLflow tracking URI         : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
Required packages           : 6 / 6
Free data-volume storage    : 8.71 GiB
Protected storage reserve   : PASS
Models loaded               : NO
Models retrained            : NO

READY FOR FINAL REPOSITORY STRUCTURE AUDIT


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-2-final-repository-structure-audit"></a>
## 2. Final Repository Structure Audit

This section inventories the completed solution before packaging and records any pre-existing packaging files. It also defines a controlled writer that preserves a checksum-named backup before replacing an unrelated existing file, preventing blind overwrites while keeping reruns idempotent.


In [2]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


REPOSITORY_AUDIT_PATH = PACKAGING_OUTPUT_ROOT / "repository_structure_audit.json"
PREEXISTING_BACKUP_ROOT = PACKAGING_OUTPUT_ROOT / "preexisting_files"
PREEXISTING_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

expected_solution_directories = (
    "api", "ui", "src", "configs", "tests", "notebooks", "reports"
)
directory_audit = {
    name: (SOLUTION_ROOT / name).is_dir()
    for name in expected_solution_directories
}

if not all(directory_audit.values()):
    raise RuntimeError(
        "Required solution directories are unavailable: "
        f"{[name for name, present in directory_audit.items() if not present]}"
    )

potential_packaging_paths = (
    SOLUTION_ROOT / "requirements.txt",
    SOLUTION_ROOT / "requirements.api.txt",
    SOLUTION_ROOT / "requirements.ui.txt",
    SOLUTION_ROOT / "Dockerfile.api",
    SOLUTION_ROOT / "Dockerfile.ui",
    SOLUTION_ROOT / "docker-compose.yaml",
    SOLUTION_ROOT / ".env.example",
    SOLUTION_ROOT / "README.md",
)


def sha256_bytes(content: bytes) -> str:
    return hashlib.sha256(content).hexdigest()


def sha256_file(path: Path) -> str:
    return sha256_bytes(path.read_bytes())


preexisting_packaging_files = [
    {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in potential_packaging_paths
    if path.is_file()
]

managed_write_events = []


def write_managed_text(path: Path, content: str, marker: str) -> str:
    # Write a generated text file while preserving unrelated prior content.
    normalized = content.strip() + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.is_file():
        existing = path.read_text(encoding="utf-8", errors="replace")

        if existing == normalized:
            action = "unchanged"
            managed_write_events.append({"path": str(path), "action": action})
            return action

        if marker not in existing:
            relative_name = str(path.relative_to(SOLUTION_ROOT)).replace("/", "__")
            backup_path = (
                PREEXISTING_BACKUP_ROOT
                / f"{relative_name}.{sha256_bytes(path.read_bytes())[:12]}.bak"
            )
            if not backup_path.exists():
                backup_path.write_bytes(path.read_bytes())

    path.write_text(normalized, encoding="utf-8")
    action = "written"
    managed_write_events.append({"path": str(path), "action": action})
    return action


repository_audit = {
    "audit_version": "final-repository-structure-audit-v1",
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "solution_root": str(SOLUTION_ROOT),
    "required_directories": directory_audit,
    "preexisting_packaging_files": preexisting_packaging_files,
    "blind_overwrite_policy": "preserve_checksum_named_backup",
}
REPOSITORY_AUDIT_PATH.write_text(
    json.dumps(repository_audit, indent=2), encoding="utf-8"
)

print("FINAL REPOSITORY STRUCTURE AUDIT")
print("-" * 110)
for directory_name, present in directory_audit.items():
    print(f"{directory_name:<30}: {'AVAILABLE' if present else 'MISSING'}")
print(f"Pre-existing packaging files : {len(preexisting_packaging_files)}")
print(f"Overwrite protection          : ENABLED")
print(f"Audit artifact                : {REPOSITORY_AUDIT_PATH}")
print(f"Models loaded                 : NO")
print()
print("READY FOR RUNTIME REQUIREMENTS")


FINAL REPOSITORY STRUCTURE AUDIT
--------------------------------------------------------------------------------------------------------------
api                           : AVAILABLE
ui                            : AVAILABLE
src                           : AVAILABLE
configs                       : AVAILABLE
tests                         : AVAILABLE
notebooks                     : AVAILABLE
reports                       : AVAILABLE
Pre-existing packaging files : 0
Overwrite protection          : ENABLED
Audit artifact                : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/packaging/repository_structure_audit.json
Models loaded                 : NO

READY FOR RUNTIME REQUIREMENTS


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-3-runtime-requirements"></a>
## 3. Runtime Requirements

This section persists separate dependency contracts for the FastAPI and Streamlit containers plus a combined reproducibility entry point. Versions are pinned to the runtime validated across Notebooks 1–8, including the CUDA-enabled PyTorch and protobuf compatibility contracts.


In [3]:
REQUIREMENTS_MARKER = "# Managed by Notebook 9"
REQUIREMENTS_PATH = SOLUTION_ROOT / "requirements.txt"
API_REQUIREMENTS_PATH = SOLUTION_ROOT / "requirements.api.txt"
UI_REQUIREMENTS_PATH = SOLUTION_ROOT / "requirements.ui.txt"

api_requirements = r'''
# Managed by Notebook 9
--extra-index-url https://download.pytorch.org/whl/cu121
torch==2.5.1+cu121
torchvision==0.20.1+cu121
numpy==1.26.4
pandas==2.2.3
Pillow==11.2.1
fastapi==0.115.12
uvicorn[standard]==0.34.0
python-multipart==0.0.20
httpx==0.23.0
PyYAML==6.0.2
transformers==4.48.3
captum==0.8.0
sentencepiece==0.2.0
accelerate==1.3.0
bitsandbytes==0.45.2
mlflow==2.15.1
protobuf==4.25.3
'''

ui_requirements = r'''
# Managed by Notebook 9
streamlit==1.41.1
httpx==0.23.0
PyYAML==6.0.2
pandas==2.2.3
Pillow==11.2.1
protobuf==4.25.3
'''

combined_requirements = r'''
# Managed by Notebook 9
-r requirements.api.txt
-r requirements.ui.txt
'''

write_managed_text(API_REQUIREMENTS_PATH, api_requirements, REQUIREMENTS_MARKER)
write_managed_text(UI_REQUIREMENTS_PATH, ui_requirements, REQUIREMENTS_MARKER)
write_managed_text(REQUIREMENTS_PATH, combined_requirements, REQUIREMENTS_MARKER)

required_pins = {
    "torch==2.5.1+cu121": API_REQUIREMENTS_PATH,
    "torchvision==0.20.1+cu121": API_REQUIREMENTS_PATH,
    "fastapi==0.115.12": API_REQUIREMENTS_PATH,
    "streamlit==1.41.1": UI_REQUIREMENTS_PATH,
    "httpx==0.23.0": UI_REQUIREMENTS_PATH,
    "protobuf==4.25.3": API_REQUIREMENTS_PATH,
}

missing_pins = [
    pin for pin, path in required_pins.items()
    if pin not in path.read_text(encoding="utf-8")
]
if missing_pins:
    raise RuntimeError(f"Required runtime pins are missing: {missing_pins}")

print("RUNTIME REQUIREMENTS")
print("-" * 100)
print(f"Combined requirements      : {REQUIREMENTS_PATH}")
print(f"FastAPI requirements       : {API_REQUIREMENTS_PATH}")
print(f"Streamlit requirements     : {UI_REQUIREMENTS_PATH}")
print(f"CUDA PyTorch contract      : 2.5.1+cu121")
print(f"TorchVision contract       : 0.20.1+cu121")
print(f"Protobuf compatibility     : 4.25.3")
print(f"Required version pins      : PASS")
print()
print("READY FOR ENVIRONMENT AND LAUNCH CONTRACTS")


RUNTIME REQUIREMENTS
----------------------------------------------------------------------------------------------------
Combined requirements      : /home/jovyan/chest-xray-ai-assistant/requirements.txt
FastAPI requirements       : /home/jovyan/chest-xray-ai-assistant/requirements.api.txt
Streamlit requirements     : /home/jovyan/chest-xray-ai-assistant/requirements.ui.txt
CUDA PyTorch contract      : 2.5.1+cu121
TorchVision contract       : 0.20.1+cu121
Protobuf compatibility     : 4.25.3
Required version pins      : PASS

READY FOR ENVIRONMENT AND LAUNCH CONTRACTS


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-4-environment-and-independent-launch-contracts"></a>
## 4. Environment and Independent Launch Contracts

This section defines configurable host, port, API-connectivity, cache, tracking, and persistent-volume settings. It also persists separate executable launch scripts and a machine-readable runtime contract for FastAPI and Streamlit.


In [4]:
ENV_MARKER = "# Managed by Notebook 9"
ENV_EXAMPLE_PATH = SOLUTION_ROOT / ".env.example"
RUNTIME_CONTRACT_PATH = CONFIG_ROOT / "runtime_contract.yaml"
API_LAUNCH_PATH = DEPLOYMENT_ROOT / "start_api.sh"
UI_LAUNCH_PATH = DEPLOYMENT_ROOT / "start_ui.sh"

environment_example = f'''
# Managed by Notebook 9
CHEST_XRAY_SOLUTION_ROOT={SOLUTION_ROOT}
CHEST_XRAY_DATA_ROOT={DATA_ROOT}
CHEST_XRAY_API_HOST=0.0.0.0
CHEST_XRAY_API_PORT=8000
CHEST_XRAY_UI_HOST=0.0.0.0
CHEST_XRAY_UI_PORT=8501
CHEST_XRAY_API_BASE_URL=http://api:8000
MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
HF_HOME={DATA_ROOT / 'hf-cache'}
HF_DATASETS_CACHE={DATA_ROOT / 'hf-cache' / 'datasets'}
TORCH_HOME={DATA_ROOT / 'models' / 'torch-cache'}
TOKENIZERS_PARALLELISM=false
'''

runtime_contract = {
    "contract_version": "cloud-native-runtime-contract-v1",
    "python_version": "3.11",
    "solution_root": str(SOLUTION_ROOT),
    "data_root": str(DATA_ROOT),
    "api": {
        "application": "api.main:app",
        "host_environment_variable": "CHEST_XRAY_API_HOST",
        "port_environment_variable": "CHEST_XRAY_API_PORT",
        "default_host": "0.0.0.0",
        "default_port": 8000,
        "health_path": "/health",
    },
    "ui": {
        "application": "ui/app.py",
        "host_environment_variable": "CHEST_XRAY_UI_HOST",
        "port_environment_variable": "CHEST_XRAY_UI_PORT",
        "api_base_url_environment_variable": "CHEST_XRAY_API_BASE_URL",
        "default_host": "0.0.0.0",
        "default_port": 8501,
        "health_path": "/_stcore/health",
        "maximum_upload_mib": 10,
    },
    "persistent_volume": {
        "host_environment_variable": "CHEST_XRAY_DATA_ROOT",
        "container_path": str(DATA_ROOT),
        "minimum_free_storage_gib": MINIMUM_FREE_STORAGE_GIB,
    },
}

api_launch = r'''
#!/usr/bin/env bash
# Managed by Notebook 9
set -euo pipefail
exec "${PYTHON_EXECUTABLE:-python}" -m uvicorn api.main:app \
  --host "${CHEST_XRAY_API_HOST:-0.0.0.0}" \
  --port "${CHEST_XRAY_API_PORT:-8000}"
'''

ui_launch = r'''
#!/usr/bin/env bash
# Managed by Notebook 9
set -euo pipefail
exec "${PYTHON_EXECUTABLE:-python}" -m streamlit run ui/app.py \
  --server.address "${CHEST_XRAY_UI_HOST:-0.0.0.0}" \
  --server.port "${CHEST_XRAY_UI_PORT:-8501}" \
  --server.maxUploadSize 10 \
  --server.headless true \
  --server.fileWatcherType none \
  --browser.gatherUsageStats false
'''

write_managed_text(ENV_EXAMPLE_PATH, environment_example, ENV_MARKER)
write_managed_text(
    RUNTIME_CONTRACT_PATH,
    "# Managed by Notebook 9\n" + yaml.safe_dump(runtime_contract, sort_keys=False),
    "# Managed by Notebook 9",
)
write_managed_text(API_LAUNCH_PATH, api_launch, "# Managed by Notebook 9")
write_managed_text(UI_LAUNCH_PATH, ui_launch, "# Managed by Notebook 9")
API_LAUNCH_PATH.chmod(0o755)
UI_LAUNCH_PATH.chmod(0o755)

persisted_runtime_contract = yaml.safe_load(
    RUNTIME_CONTRACT_PATH.read_text(encoding="utf-8")
)
if persisted_runtime_contract["api"]["application"] != "api.main:app":
    raise RuntimeError("FastAPI launch target is not preserved.")
if persisted_runtime_contract["ui"]["api_base_url_environment_variable"] != "CHEST_XRAY_API_BASE_URL":
    raise RuntimeError("UI-to-API connectivity contract is not preserved.")

print("ENVIRONMENT AND INDEPENDENT LAUNCH CONTRACTS")
print("-" * 110)
print(f"Environment template       : {ENV_EXAMPLE_PATH}")
print(f"Runtime contract           : {RUNTIME_CONTRACT_PATH}")
print(f"FastAPI launch script      : {API_LAUNCH_PATH}")
print(f"Streamlit launch script    : {UI_LAUNCH_PATH}")
print(f"FastAPI application        : api.main:app")
print(f"UI backend environment     : CHEST_XRAY_API_BASE_URL")
print(f"Persistent data path       : {DATA_ROOT}")
print()
print("READY FOR FASTAPI CONTAINER PACKAGING")


ENVIRONMENT AND INDEPENDENT LAUNCH CONTRACTS
--------------------------------------------------------------------------------------------------------------
Environment template       : /home/jovyan/chest-xray-ai-assistant/.env.example
Runtime contract           : /home/jovyan/chest-xray-ai-assistant/configs/runtime_contract.yaml
FastAPI launch script      : /home/jovyan/chest-xray-ai-assistant/deployment/start_api.sh
Streamlit launch script    : /home/jovyan/chest-xray-ai-assistant/deployment/start_ui.sh
FastAPI application        : api.main:app
UI backend environment     : CHEST_XRAY_API_BASE_URL
Persistent data path       : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data

READY FOR FASTAPI CONTAINER PACKAGING


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-5-fastapi-container-packaging"></a>
## 5. FastAPI Container Packaging

This section creates the GPU-capable FastAPI image contract. The container preserves Python 3.11, CUDA 12.1, the canonical solution and data paths, persistent caches, the authoritative `api.main:app` target, and a bounded health check.


In [5]:
DOCKERFILE_API_PATH = SOLUTION_ROOT / "Dockerfile.api"
dockerfile_api = f'''
# Managed by Notebook 9
FROM pytorch/pytorch:2.5.1-cuda12.1-cudnn9-runtime

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    TOKENIZERS_PARALLELISM=false \\
    CHEST_XRAY_SOLUTION_ROOT={SOLUTION_ROOT} \\
    CHEST_XRAY_DATA_ROOT={DATA_ROOT} \\
    HF_HOME={DATA_ROOT / 'hf-cache'} \\
    HF_DATASETS_CACHE={DATA_ROOT / 'hf-cache' / 'datasets'} \\
    TORCH_HOME={DATA_ROOT / 'models' / 'torch-cache'} \\
    MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}

WORKDIR {SOLUTION_ROOT}
COPY requirements.api.txt /tmp/requirements.api.txt
RUN python -m pip install --no-cache-dir -r /tmp/requirements.api.txt
COPY . {SOLUTION_ROOT}

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=10s --start-period=180s --retries=5 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5)" || exit 1

CMD ["python", "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
write_managed_text(DOCKERFILE_API_PATH, dockerfile_api, "# Managed by Notebook 9")

api_docker_text = DOCKERFILE_API_PATH.read_text(encoding="utf-8")
for required_fragment in (
    "pytorch/pytorch:2.5.1-cuda12.1-cudnn9-runtime",
    "api.main:app",
    "/health",
    str(DATA_ROOT),
):
    if required_fragment not in api_docker_text:
        raise RuntimeError(f"FastAPI Dockerfile is missing: {required_fragment}")

print("FASTAPI CONTAINER PACKAGING")
print("-" * 100)
print(f"Dockerfile                : {DOCKERFILE_API_PATH}")
print(f"Base runtime              : PyTorch 2.5.1 / CUDA 12.1 / cuDNN 9")
print(f"Application target        : api.main:app")
print(f"Exposed port              : 8000")
print(f"Health path               : /health")
print(f"Persistent data path      : {DATA_ROOT}")
print(f"Models built or downloaded: NO")
print()
print("READY FOR STREAMLIT CONTAINER PACKAGING")


FASTAPI CONTAINER PACKAGING
----------------------------------------------------------------------------------------------------
Dockerfile                : /home/jovyan/chest-xray-ai-assistant/Dockerfile.api
Base runtime              : PyTorch 2.5.1 / CUDA 12.1 / cuDNN 9
Application target        : api.main:app
Exposed port              : 8000
Health path               : /health
Persistent data path      : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
Models built or downloaded: NO

READY FOR STREAMLIT CONTAINER PACKAGING


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-6-streamlit-container-packaging"></a>
## 6. Streamlit Container Packaging

This section creates the lean Streamlit image contract. It installs only UI dependencies, exposes port 8501, retains the 10 MiB upload boundary, uses the Streamlit health endpoint, and receives the FastAPI address through `CHEST_XRAY_API_BASE_URL`.


In [6]:
DOCKERFILE_UI_PATH = SOLUTION_ROOT / "Dockerfile.ui"
dockerfile_ui = f'''
# Managed by Notebook 9
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    CHEST_XRAY_SOLUTION_ROOT={SOLUTION_ROOT} \\
    CHEST_XRAY_API_BASE_URL=http://api:8000

WORKDIR {SOLUTION_ROOT}
COPY requirements.ui.txt /tmp/requirements.ui.txt
RUN python -m pip install --no-cache-dir -r /tmp/requirements.ui.txt
COPY . {SOLUTION_ROOT}

EXPOSE 8501
HEALTHCHECK --interval=30s --timeout=10s --start-period=20s --retries=5 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=5)" || exit 1

CMD ["python", "-m", "streamlit", "run", "ui/app.py", "--server.address", "0.0.0.0", "--server.port", "8501", "--server.maxUploadSize", "10", "--server.headless", "true", "--server.fileWatcherType", "none", "--browser.gatherUsageStats", "false"]
'''
write_managed_text(DOCKERFILE_UI_PATH, dockerfile_ui, "# Managed by Notebook 9")

ui_docker_text = DOCKERFILE_UI_PATH.read_text(encoding="utf-8")
for required_fragment in (
    "python:3.11-slim",
    "ui/app.py",
    "CHEST_XRAY_API_BASE_URL=http://api:8000",
    "/_stcore/health",
    "--server.maxUploadSize",
):
    if required_fragment not in ui_docker_text:
        raise RuntimeError(f"Streamlit Dockerfile is missing: {required_fragment}")

print("STREAMLIT CONTAINER PACKAGING")
print("-" * 100)
print(f"Dockerfile              : {DOCKERFILE_UI_PATH}")
print(f"Base runtime            : Python 3.11 slim")
print(f"Application             : ui/app.py")
print(f"Exposed port            : 8501")
print(f"Health path             : /_stcore/health")
print(f"Backend address         : http://api:8000")
print(f"Upload limit            : 10 MiB")
print(f"Model dependencies      : NOT INCLUDED")
print()
print("READY FOR COMPOSE DEPLOYMENT CONFIGURATION")


STREAMLIT CONTAINER PACKAGING
----------------------------------------------------------------------------------------------------
Dockerfile              : /home/jovyan/chest-xray-ai-assistant/Dockerfile.ui
Base runtime            : Python 3.11 slim
Application             : ui/app.py
Exposed port            : 8501
Health path             : /_stcore/health
Backend address         : http://api:8000
Upload limit            : 10 MiB
Model dependencies      : NOT INCLUDED

READY FOR COMPOSE DEPLOYMENT CONFIGURATION


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-7-compose-connectivity-and-persistent-volume-configuration"></a>
## 7. Compose Connectivity and Persistent-Volume Configuration

This section connects the two independent containers. FastAPI receives GPU access and the persistent data volume; Streamlit waits for API health and communicates only through the internal service URL. Both services expose configurable host ports and retain independent health checks.


In [7]:
COMPOSE_PATH = SOLUTION_ROOT / "docker-compose.yaml"
compose_document = {
    "services": {
        "api": {
            "build": {"context": ".", "dockerfile": "Dockerfile.api"},
            "image": "chest-xray-assistant-api:latest",
            "environment": {
                "CHEST_XRAY_API_HOST": "0.0.0.0",
                "CHEST_XRAY_API_PORT": "8000",
                "CHEST_XRAY_DATA_ROOT": str(DATA_ROOT),
                "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
                "HF_HOME": str(DATA_ROOT / "hf-cache"),
                "HF_DATASETS_CACHE": str(DATA_ROOT / "hf-cache" / "datasets"),
                "TORCH_HOME": str(DATA_ROOT / "models" / "torch-cache"),
                "TOKENIZERS_PARALLELISM": "false",
            },
            "ports": ["${CHEST_XRAY_API_PORT:-8000}:8000"],
            "volumes": [f"${{CHEST_XRAY_DATA_ROOT:-{DATA_ROOT}}}:{DATA_ROOT}"],
            "healthcheck": {
                "test": ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5)"],
                "interval": "30s", "timeout": "10s", "retries": 5,
                "start_period": "180s",
            },
            "deploy": {
                "resources": {"reservations": {"devices": [
                    {"driver": "nvidia", "count": "all", "capabilities": ["gpu"]}
                ]}}
            },
            "restart": "unless-stopped",
            "networks": ["chest-xray-network"],
        },
        "ui": {
            "build": {"context": ".", "dockerfile": "Dockerfile.ui"},
            "image": "chest-xray-assistant-ui:latest",
            "environment": {
                "CHEST_XRAY_API_BASE_URL": "http://api:8000",
                "CHEST_XRAY_UI_HOST": "0.0.0.0",
                "CHEST_XRAY_UI_PORT": "8501",
            },
            "ports": ["${CHEST_XRAY_UI_PORT:-8501}:8501"],
            "depends_on": {"api": {"condition": "service_healthy"}},
            "healthcheck": {
                "test": ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=5)"],
                "interval": "30s", "timeout": "10s", "retries": 5,
                "start_period": "20s",
            },
            "restart": "unless-stopped",
            "networks": ["chest-xray-network"],
        },
    },
    "networks": {"chest-xray-network": {"driver": "bridge"}},
}

compose_text = (
    "# Managed by Notebook 9\n"
    + yaml.safe_dump(compose_document, sort_keys=False, width=120)
)
write_managed_text(COMPOSE_PATH, compose_text, "# Managed by Notebook 9")

persisted_compose = yaml.safe_load(COMPOSE_PATH.read_text(encoding="utf-8"))
api_service = persisted_compose["services"]["api"]
ui_service = persisted_compose["services"]["ui"]

if ui_service["environment"]["CHEST_XRAY_API_BASE_URL"] != "http://api:8000":
    raise RuntimeError("Compose does not preserve internal API connectivity.")
if "volumes" not in api_service or "deploy" not in api_service:
    raise RuntimeError("Compose does not preserve the persistent volume and GPU contracts.")

print("COMPOSE CONNECTIVITY AND PERSISTENT-VOLUME CONFIGURATION")
print("-" * 110)
print(f"Compose file               : {COMPOSE_PATH}")
print(f"Services                   : api, ui")
print(f"Internal API URL           : http://api:8000")
print(f"API host port              : configurable, default 8000")
print(f"UI host port               : configurable, default 8501")
print(f"Persistent data volume     : INCLUDED")
print(f"GPU reservation            : INCLUDED")
print(f"Health-gated UI dependency : INCLUDED")
print()
print("READY FOR DEPLOYMENT HEALTH VALIDATION CONTRACT")


COMPOSE CONNECTIVITY AND PERSISTENT-VOLUME CONFIGURATION
--------------------------------------------------------------------------------------------------------------
Compose file               : /home/jovyan/chest-xray-ai-assistant/docker-compose.yaml
Services                   : api, ui
Internal API URL           : http://api:8000
API host port              : configurable, default 8000
UI host port               : configurable, default 8501
Persistent data volume     : INCLUDED
GPU reservation            : INCLUDED
Health-gated UI dependency : INCLUDED

READY FOR DEPLOYMENT HEALTH VALIDATION CONTRACT


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-8-deployment-health-validation-contract"></a>
## 8. Deployment Health Validation Contract

This section persists a standalone post-deployment validator for FastAPI health, OpenAPI path count, Streamlit health, and root HTML. The validator uses HTTP only and can be executed after local, Compose, or externally orchestrated deployment.


In [8]:
import py_compile

DEPLOYMENT_VALIDATOR_PATH = DEPLOYMENT_ROOT / "validate_services.py"
deployment_validator = r'''
# Managed by Notebook 9
from __future__ import annotations

import json
import os
import urllib.request

api_url = os.getenv("CHEST_XRAY_API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
ui_url = os.getenv("CHEST_XRAY_UI_BASE_URL", "http://127.0.0.1:8501").rstrip("/")

def read(url: str) -> tuple[int, str, str]:
    with urllib.request.urlopen(url, timeout=30) as response:
        return response.status, response.headers.get("content-type", ""), response.read().decode("utf-8")

api_status, _, api_body = read(f"{api_url}/health")
openapi_status, _, openapi_body = read(f"{api_url}/openapi.json")
ui_status, _, ui_health = read(f"{ui_url}/_stcore/health")
ui_root_status, ui_root_type, _ = read(ui_url)
openapi = json.loads(openapi_body)

checks = {
    "fastapi_health": api_status == 200 and json.loads(api_body).get("status") == "success",
    "openapi_paths": openapi_status == 200 and len(openapi.get("paths", {})) == 12,
    "streamlit_health": ui_status == 200 and ui_health.strip().lower() == "ok",
    "streamlit_root": ui_root_status == 200 and "text/html" in ui_root_type.lower(),
}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise RuntimeError(f"Deployment health validation failed: {failed}")
print(json.dumps({"status": "pass", "checks": checks}, indent=2))
'''

write_managed_text(
    DEPLOYMENT_VALIDATOR_PATH,
    deployment_validator,
    "# Managed by Notebook 9",
)
py_compile.compile(str(DEPLOYMENT_VALIDATOR_PATH), doraise=True)

validator_text = DEPLOYMENT_VALIDATOR_PATH.read_text(encoding="utf-8")
for required_endpoint in ("/health", "/openapi.json", "/_stcore/health"):
    if required_endpoint not in validator_text:
        raise RuntimeError(f"Deployment validator is missing {required_endpoint}.")

print("DEPLOYMENT HEALTH VALIDATION CONTRACT")
print("-" * 100)
print(f"Validator module          : {DEPLOYMENT_VALIDATOR_PATH}")
print(f"FastAPI health            : INCLUDED")
print(f"OpenAPI path count        : 12")
print(f"Streamlit health          : INCLUDED")
print(f"Streamlit root HTML       : INCLUDED")
print(f"Backend boundary          : HTTP ONLY")
print(f"Syntax compilation        : PASS")
print()
print("READY FOR SOLUTION README")


DEPLOYMENT HEALTH VALIDATION CONTRACT
----------------------------------------------------------------------------------------------------
Validator module          : /home/jovyan/chest-xray-ai-assistant/deployment/validate_services.py
FastAPI health            : INCLUDED
OpenAPI path count        : 12
Streamlit health          : INCLUDED
Streamlit root HTML       : INCLUDED
Backend boundary          : HTTP ONLY
Syntax compilation        : PASS

READY FOR SOLUTION README


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-9-solution-readme"></a>
## 9. Solution README

This section creates the primary solution README with purpose, architecture, capabilities, prerequisites, quick start, service URLs, safety boundary, repository structure, validation evidence, and links to the detailed documentation produced in subsequent sections.


In [9]:
README_PATH = SOLUTION_ROOT / "README.md"
README_MARKER = "<!-- Managed by Notebook 9 -->"
readme_text = f'''
<!-- Managed by Notebook 9 -->
# API-Driven Chest X-Ray Analysis and Explanation Assistant using ChestMNIST

An educational decision-support prototype that combines a frozen ResNet-18 ChestMNIST multilabel classifier, Grad-CAM visual evidence, grounded FLAN-T5 language generation, a versioned FastAPI backend, and a Streamlit interface.

> **Educational-use limitation:** This output is generated by an educational decision-support prototype. It is not a diagnosis and should not replace review by a qualified healthcare professional.

## Solution workflow

1. Validate and decode a PNG or JPEG image.
2. Run frozen multilabel classification and compare frozen per-label thresholds.
3. Produce cautious no-target or crossed-finding interpretation.
4. Generate Grad-CAM evidence only for crossed findings.
5. Generate structured, grounded language outputs and guarded follow-up answers.
6. Deliver the workflow through FastAPI and an HTTP-only Streamlit interface.
7. Preserve model, prompt, API, UI, test, and operational lineage in MLflow and checksum registries.

## Architecture

```mermaid
flowchart LR
    U[User] --> S[Streamlit UI]
    S -->|HTTP| A[FastAPI]
    A --> C[Frozen CV + Grad-CAM]
    A --> L[Grounded language + guardrails]
    A --> P[Prediction store]
    A --> M[MLflow + evidence]
```

The Streamlit container never imports or loads the computer-vision model, language model, Grad-CAM service, guardrail, or prediction store.

## Quick start with Docker Compose

```bash
cp .env.example .env
docker compose build
docker compose up -d
python deployment/validate_services.py
```

- Streamlit: `http://127.0.0.1:8501`
- FastAPI health: `http://127.0.0.1:8000/health`
- OpenAPI: `http://127.0.0.1:8000/docs`

The host data root must already contain the persisted models, caches, contracts, thresholds, prompt registry, MLflow tracking store, and output evidence produced by Notebooks 1–8.

## Repository structure

- `api/` — FastAPI routes, schemas, configuration, and application factory
- `ui/` — Streamlit application, components, and HTTP client
- `src/` — frozen model, language, explainability, workflow, and guardrail services
- `configs/` — path, model, prompt, API, UI, and runtime contracts
- `tests/` — reusable API and focused interface tests
- `deployment/` — launch scripts and post-deployment validation
- `docs/` — architecture, usage, API, safety, testing, and reproducibility guides
- `reports/` — demonstration inputs and solution evidence

## Documentation

- [Architecture](docs/architecture.md)
- [Usage and deployment](docs/usage.md)
- [API examples](docs/api_examples.md)
- [Safety and limitations](docs/safety_and_limitations.md)
- [Testing, MLflow, and reproducibility](docs/testing_mlflow_reproducibility.md)
- [Demonstration evidence](docs/demonstration_evidence.md)

## Validation boundary

The packaging notebook performs static Docker/Compose validation and preserves the already successful independent FastAPI, Streamlit, client-contract, UI-state, live-workflow, screenshot-readiness, MLflow, and storage evidence. Building or publishing container images is an explicit deployment action and is not performed automatically by the notebook.
'''

write_managed_text(README_PATH, readme_text, README_MARKER)
persisted_readme = README_PATH.read_text(encoding="utf-8")
required_readme_sections = (
    "## Solution workflow", "## Architecture", "## Quick start with Docker Compose",
    "## Repository structure", "## Documentation", "## Validation boundary",
)
missing_readme_sections = [
    section for section in required_readme_sections if section not in persisted_readme
]
if missing_readme_sections:
    raise RuntimeError(f"README sections are missing: {missing_readme_sections}")

print("SOLUTION README")
print("-" * 100)
print(f"README path              : {README_PATH}")
print(f"Required sections        : {len(required_readme_sections)} / {len(required_readme_sections)}")
print(f"Quick-start contract     : INCLUDED")
print(f"Architecture overview    : INCLUDED")
print(f"Educational limitation  : PRESERVED")
print(f"Container build action   : NOT EXECUTED")
print()
print("READY FOR ARCHITECTURE DOCUMENTATION")


SOLUTION README
----------------------------------------------------------------------------------------------------
README path              : /home/jovyan/chest-xray-ai-assistant/README.md
Required sections        : 6 / 6
Quick-start contract     : INCLUDED
Architecture overview    : INCLUDED
Educational limitation  : PRESERVED
Container build action   : NOT EXECUTED

READY FOR ARCHITECTURE DOCUMENTATION


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-10-architecture-documentation"></a>
## 10. Architecture Documentation

This section documents component ownership, request flow, trust boundaries, persistence, health contracts, and the separation between packaging-time validation and deployment-time execution.


In [10]:
ARCHITECTURE_PATH = DOCS_ROOT / "architecture.md"
architecture_text = f'''
<!-- Managed by Notebook 9 -->
# Solution Architecture

## Runtime topology

```mermaid
flowchart TB
    B[Browser] -->|HTTP 8501| UI[Streamlit container]
    UI -->|HTTP 8000| API[FastAPI container]
    API --> CV[Frozen ResNet-18]
    API --> GC[LayerGradCam]
    API --> NLP[Grounded FLAN-T5]
    API --> PS[In-memory prediction store]
    API --> PV[(Persistent data volume)]
    API --> MF[(MLflow store)]
```

## Ownership boundaries

| Component | Owns | Must not own |
|---|---|---|
| Streamlit | upload UX, preview, HTTP requests, rendering, session state | models, thresholds, Grad-CAM, guardrails, prediction storage |
| FastAPI | schemas, validation, workflows, controlled errors, operational metrics | browser session state |
| CV service | frozen probabilities and threshold decisions | clinical interpretation |
| Explainability service | attribution evidence for crossed findings | lesion or anatomical confirmation |
| Language service | evidence-grounded educational text | image inspection or unsupported medical advice |
| Persistent volume | models, caches, MLflow, artifacts, evidence | transient container layers |

## Primary request sequence

```mermaid
sequenceDiagram
    participant U as User
    participant S as Streamlit
    participant A as FastAPI
    participant W as Workflow services
    U->>S: Upload image and submit
    S->>A: POST /api/v1/analyze-complete
    A->>W: Validate, classify, explain, generate
    W-->>A: Structured evidence
    A-->>S: CompleteAnalysisResponse
    S-->>U: Findings, evidence, language, lineage
```

## Deployment contracts

- API target: `api.main:app`, port `8000`, health `/health`.
- UI target: `ui/app.py`, port `8501`, health `/_stcore/health`.
- Internal UI-to-API URL: `http://api:8000`.
- Persistent host volume: `{DATA_ROOT}` mounted at the same container path.
- API requests may use NVIDIA GPU resources; the UI image is CPU-only.
- MLflow tracking URI: `{MLFLOW_TRACKING_URI}`.

## Safety boundary

Grad-CAM is attribution evidence, not segmentation or confirmation. No-target-finding means that none of the fourteen supported labels crossed its frozen threshold; it does not establish clinical normality. All output requires qualified professional review.
'''
write_managed_text(ARCHITECTURE_PATH, architecture_text, "<!-- Managed by Notebook 9 -->")

architecture_doc = ARCHITECTURE_PATH.read_text(encoding="utf-8")
for phrase in ("Streamlit container", "FastAPI container", "Persistent data volume", "Safety boundary"):
    if phrase not in architecture_doc:
        raise RuntimeError(f"Architecture documentation is missing: {phrase}")

print("ARCHITECTURE DOCUMENTATION")
print("-" * 100)
print(f"Architecture document     : {ARCHITECTURE_PATH}")
print(f"Runtime topology          : INCLUDED")
print(f"Ownership boundaries      : INCLUDED")
print(f"Primary request sequence  : INCLUDED")
print(f"Deployment contracts      : INCLUDED")
print(f"Safety boundary           : INCLUDED")
print()
print("READY FOR USAGE AND DEPLOYMENT INSTRUCTIONS")


ARCHITECTURE DOCUMENTATION
----------------------------------------------------------------------------------------------------
Architecture document     : /home/jovyan/chest-xray-ai-assistant/docs/architecture.md
Runtime topology          : INCLUDED
Ownership boundaries      : INCLUDED
Primary request sequence  : INCLUDED
Deployment contracts      : INCLUDED
Safety boundary           : INCLUDED

READY FOR USAGE AND DEPLOYMENT INSTRUCTIONS


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-11-usage-and-deployment-instructions"></a>
## 11. Usage and Deployment Instructions

This section documents prerequisite artifacts, environment preparation, independent local launch, Compose deployment, service validation, shutdown, and troubleshooting without embedding credentials or rebuilding model artifacts.


In [11]:
USAGE_PATH = DOCS_ROOT / "usage.md"
usage_text = f'''
<!-- Managed by Notebook 9 -->
# Usage and Deployment

## Prerequisites

- Linux host with Docker Engine and Docker Compose v2.
- NVIDIA driver and NVIDIA Container Toolkit for GPU-backed API deployment.
- At least 4 GiB free on the persistent data volume.
- Completed Notebook 8 readiness artifact at `{NOTEBOOK8_READINESS_PATH}`.
- Persisted models, thresholds, prompt registry, caches, MLflow store, and configuration beneath `{DATA_ROOT}` and `{SOLUTION_ROOT}`.

## Configure

```bash
cd {SOLUTION_ROOT}
cp .env.example .env
# Review CHEST_XRAY_DATA_ROOT and exposed ports before deployment.
```

Do not put credentials, access tokens, patient information, or private image paths in `.env`.

## Independent local launch

Terminal 1:
```bash
./deployment/start_api.sh
```

Terminal 2:
```bash
CHEST_XRAY_API_BASE_URL=http://127.0.0.1:8000 ./deployment/start_ui.sh
```

Validate:
```bash
python deployment/validate_services.py
```

## Docker Compose

```bash
docker compose config
docker compose build
docker compose up -d
docker compose ps
python deployment/validate_services.py
```

Open `http://127.0.0.1:8501`, upload a PNG or JPEG no larger than 10 MiB, review the educational limitation, and explicitly start complete analysis.

## Stop services

```bash
docker compose down
```

The persistent data volume is a host bind mount and is not removed by `docker compose down`.

## Troubleshooting

- API startup may take longer while frozen models are loaded; retain the configured health-check start period.
- Confirm NVIDIA Container Toolkit availability when the API cannot access CUDA.
- Confirm `CHEST_XRAY_DATA_ROOT` points to the completed persistent data hierarchy.
- Confirm the UI uses `http://api:8000` inside Compose, not `127.0.0.1:8000`.
- Inspect `docker compose logs api` and `docker compose logs ui` without exposing uploaded images or credentials.
'''
write_managed_text(USAGE_PATH, usage_text, "<!-- Managed by Notebook 9 -->")

usage_doc = USAGE_PATH.read_text(encoding="utf-8")
for section in ("## Prerequisites", "## Configure", "## Independent local launch", "## Docker Compose", "## Stop services", "## Troubleshooting"):
    if section not in usage_doc:
        raise RuntimeError(f"Usage documentation is missing: {section}")

print("USAGE AND DEPLOYMENT INSTRUCTIONS")
print("-" * 100)
print(f"Usage document           : {USAGE_PATH}")
print(f"Prerequisites            : INCLUDED")
print(f"Independent launch       : INCLUDED")
print(f"Compose deployment       : INCLUDED")
print(f"Service validation       : INCLUDED")
print(f"Safe shutdown            : INCLUDED")
print(f"Troubleshooting          : INCLUDED")
print()
print("READY FOR API EXAMPLES")


USAGE AND DEPLOYMENT INSTRUCTIONS
----------------------------------------------------------------------------------------------------
Usage document           : /home/jovyan/chest-xray-ai-assistant/docs/usage.md
Prerequisites            : INCLUDED
Independent launch       : INCLUDED
Compose deployment       : INCLUDED
Service validation       : INCLUDED
Safe shutdown            : INCLUDED
Troubleshooting          : INCLUDED

READY FOR API EXAMPLES


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-12-api-examples"></a>
## 12. API Examples

This section documents copyable health, model-information, complete-analysis, grounded-question, stored-prediction, and operational-metrics requests using the exact persisted endpoint and payload contracts.


In [12]:
API_EXAMPLES_PATH = DOCS_ROOT / "api_examples.md"
api_examples_text = r'''
<!-- Managed by Notebook 9 -->
# API Examples

Set the backend address:
```bash
export API_BASE_URL=http://127.0.0.1:8000
```

## Health
```bash
curl --fail "$API_BASE_URL/health"
```

## Model information and metrics
```bash
curl --fail "$API_BASE_URL/api/v1/model/info"
curl --fail "$API_BASE_URL/api/v1/model/metrics"
```

## Complete analysis
The authoritative multipart image field is `image`; `question` is optional.
```bash
curl --fail -X POST "$API_BASE_URL/api/v1/analyze-complete" \
  -H "Accept: application/json" \
  -F "image=@reports/streamlit_demo_input.png;type=image/png" \
  -F "question=What does the threshold result mean?"
```

Preserve the returned `prediction_id` for follow-up and retrieval.

## Grounded follow-up question
```bash
curl --fail -X POST "$API_BASE_URL/api/v1/question/answer" \
  -H "Content-Type: application/json" \
  -d '{"prediction_id":"REPLACE_WITH_UUID","question":"What does this result mean?"}'
```

The client cannot submit probabilities, findings, thresholds, image evidence, or any other grounding context.

## Stored prediction
```bash
curl --fail "$API_BASE_URL/api/v1/predictions/REPLACE_WITH_UUID"
```

## Operational and LLMOps metrics
```bash
curl --fail "$API_BASE_URL/api/v1/llmops/metrics"
```

## Controlled errors
Error responses expose the versioned `APIErrorResponse` fields and must be handled as structured JSON. Do not display backend tracebacks, internal paths, or raw exception objects to interface users.
'''
write_managed_text(API_EXAMPLES_PATH, api_examples_text, "<!-- Managed by Notebook 9 -->")

persisted_api_examples = API_EXAMPLES_PATH.read_text(encoding="utf-8")
required_api_contracts = (
    "/health", "/api/v1/model/info", "/api/v1/model/metrics",
    "/api/v1/analyze-complete", 'image=@',
    "/api/v1/question/answer", "prediction_id",
    "/api/v1/predictions/", "/api/v1/llmops/metrics",
)
missing_examples = [item for item in required_api_contracts if item not in persisted_api_examples]
if missing_examples:
    raise RuntimeError(f"API examples are incomplete: {missing_examples}")

print("API EXAMPLES")
print("-" * 100)
print(f"API examples document      : {API_EXAMPLES_PATH}")
print(f"Health and system calls    : INCLUDED")
print(f"Complete-analysis field    : image")
print(f"Grounded-question fields   : prediction_id, question")
print(f"Stored retrieval           : INCLUDED")
print(f"Operational metrics        : INCLUDED")
print(f"Client grounding context   : NOT EXPOSED")
print()
print("READY FOR SAFETY AND LIMITATIONS DOCUMENTATION")


API EXAMPLES
----------------------------------------------------------------------------------------------------
API examples document      : /home/jovyan/chest-xray-ai-assistant/docs/api_examples.md
Health and system calls    : INCLUDED
Complete-analysis field    : image
Grounded-question fields   : prediction_id, question
Stored retrieval           : INCLUDED
Operational metrics        : INCLUDED
Client grounding context   : NOT EXPOSED

READY FOR SAFETY AND LIMITATIONS DOCUMENTATION


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-13-safety-and-limitations-documentation"></a>
## 13. Safety and Limitations Documentation

This section centralizes the exact educational limitation, supported and unsupported claims, Grad-CAM interpretation boundary, no-target-finding wording, privacy constraints, and qualified-review requirement for deployment and demonstration.


In [13]:
SAFETY_PATH = DOCS_ROOT / "safety_and_limitations.md"
EDUCATIONAL_LIMITATION = (
    "This output is generated by an educational decision-support prototype. "
    "It is not a diagnosis and should not replace review by a qualified "
    "healthcare professional."
)
safety_text = f'''
<!-- Managed by Notebook 9 -->
# Safety and Limitations

> **{EDUCATIONAL_LIMITATION}**

## Supported purpose

The solution demonstrates API-driven orchestration of a frozen ChestMNIST multilabel model, frozen decision thresholds, Grad-CAM attribution evidence, grounded educational language generation, controlled follow-up questions, operational metrics, and versioned lineage.

## Prohibited interpretation

The solution must not be presented as:

- a clinical diagnostic system;
- a substitute for a radiologist, physician, or other qualified professional;
- confirmation that a lesion or anatomical abnormality exists;
- proof that a no-target-finding image is clinically normal;
- an independent source of medication, treatment, or emergency advice.

## Finding boundary

The model evaluates only fourteen ChestMNIST target labels. A crossed threshold is a frozen model decision, not a diagnosis. A no-target-finding result means only that none of those fourteen labels crossed its frozen threshold.

## Grad-CAM boundary

Grad-CAM highlights regions that influenced a model output. It does not provide segmentation, causal proof, anatomical confirmation, lesion confirmation, measurement, or clinical diagnosis.

## Grounded-language boundary

Language output is grounded only in structured API evidence, approved descriptions, frozen thresholds, model lineage, explainability metadata, and explicit safety limitations. The language model does not inspect the image independently and must not invent findings, symptoms, measurements, diagnoses, or treatment guidance.

## Privacy and operational use

Use only authorized, de-identified demonstration images. Do not place personal health information, credentials, tokens, or private image paths in logs, screenshots, environment files, MLflow parameters, or artifact registries. The in-memory prediction store is process-local and is not a clinical record system.
'''
write_managed_text(SAFETY_PATH, safety_text, "<!-- Managed by Notebook 9 -->")

persisted_safety = SAFETY_PATH.read_text(encoding="utf-8")
safety_requirements = (
    EDUCATIONAL_LIMITATION, "not a diagnosis", "no-target-finding",
    "does not provide segmentation", "does not inspect the image independently",
    "de-identified demonstration images",
)
missing_safety = [item for item in safety_requirements if item not in persisted_safety]
if missing_safety:
    raise RuntimeError(f"Safety documentation is incomplete: {missing_safety}")

print("SAFETY AND LIMITATIONS DOCUMENTATION")
print("-" * 100)
print(f"Safety document           : {SAFETY_PATH}")
print(f"Educational limitation    : PRESERVED")
print(f"No-target boundary        : PRESERVED")
print(f"Grad-CAM boundary         : PRESERVED")
print(f"Grounded-language boundary: PRESERVED")
print(f"Privacy guidance          : INCLUDED")
print(f"Clinical claims           : NOT USED")
print()
print("READY FOR TESTING, MLFLOW, AND REPRODUCIBILITY INSTRUCTIONS")


SAFETY AND LIMITATIONS DOCUMENTATION
----------------------------------------------------------------------------------------------------
Safety document           : /home/jovyan/chest-xray-ai-assistant/docs/safety_and_limitations.md
Educational limitation    : PRESERVED
No-target boundary        : PRESERVED
Grad-CAM boundary         : PRESERVED
Grounded-language boundary: PRESERVED
Privacy guidance          : INCLUDED
Clinical claims           : NOT USED

READY FOR TESTING, MLFLOW, AND REPRODUCIBILITY INSTRUCTIONS


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-14-testing-mlflow-and-reproducibility-instructions"></a>
## 14. Testing, MLflow, and Reproducibility Instructions

This section documents focused test commands, post-deployment health validation, MLflow inspection, versioned evidence locations, clean-environment reproduction order, and the boundary against retraining or redownloading persisted models.


In [14]:
TESTING_MLFLOW_PATH = DOCS_ROOT / "testing_mlflow_reproducibility.md"
testing_mlflow_text = f'''
<!-- Managed by Notebook 9 -->
# Testing, MLflow, and Reproducibility

## Reusable tests

```bash
cd {SOLUTION_ROOT}
python -m pytest tests/api -q
```

Notebook 8 additionally persists focused HTTP-client and Streamlit state/component evidence. Pytest process exit code `0` is authoritative; do not compare executed cases with raw `test_...` function counts because parameterization changes case counts.

## Post-deployment validation

```bash
python deployment/validate_services.py
```

This checks FastAPI health, twelve OpenAPI paths, Streamlit health, and Streamlit root HTML through HTTP.

## MLflow

```bash
export MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
mlflow ui --host 0.0.0.0 --port 5000
```

The tracking store contains model, explainability, language, FastAPI, Streamlit, and final packaging lineage. MLflow may emit a non-blocking `pkg_resources` deprecation warning; do not change the validated environment solely for that warning.

## Evidence locations

- API readiness: `{API_OUTPUT_ROOT / 'api_integration_readiness.json'}`
- API registry: `{API_OUTPUT_ROOT / 'api_artifact_registry.json'}`
- Streamlit readiness: `{NOTEBOOK8_READINESS_PATH}`
- UI registry: `{UI_OUTPUT_ROOT / 'ui_artifact_registry.json'}`
- Packaging evidence: `{PACKAGING_OUTPUT_ROOT}`

## Reproduction order

1. Provision Python 3.11 and the validated CUDA 12.1 GPU runtime.
2. Restore the solution repository and persistent data-volume hierarchy.
3. Install the pinned dependency contract.
4. Verify checksums and readiness artifacts from Notebooks 1–8.
5. Build the API and UI images without embedding the persistent data volume.
6. Mount the existing data volume and start FastAPI before Streamlit.
7. Run the post-deployment validator and reusable API tests.
8. Review the educational limitation and demonstration evidence.

Do not retrain models, regenerate datasets, recalculate frozen thresholds, or redownload model artifacts as part of packaging reproduction.
'''
write_managed_text(
    TESTING_MLFLOW_PATH,
    testing_mlflow_text,
    "<!-- Managed by Notebook 9 -->",
)

persisted_testing_doc = TESTING_MLFLOW_PATH.read_text(encoding="utf-8")
for phrase in ("python -m pytest tests/api -q", "MLFLOW_TRACKING_URI", "Reproduction order", "Do not retrain models"):
    if phrase not in persisted_testing_doc:
        raise RuntimeError(f"Testing and reproducibility documentation is missing: {phrase}")

print("TESTING, MLFLOW, AND REPRODUCIBILITY INSTRUCTIONS")
print("-" * 110)
print(f"Documentation             : {TESTING_MLFLOW_PATH}")
print(f"Reusable API tests        : INCLUDED")
print(f"Post-deployment validation: INCLUDED")
print(f"MLflow usage              : INCLUDED")
print(f"Evidence locations        : INCLUDED")
print(f"Reproduction order        : INCLUDED")
print(f"Retraining                : NOT REQUIRED")
print()
print("READY FOR STATIC PACKAGING VALIDATION")


TESTING, MLFLOW, AND REPRODUCIBILITY INSTRUCTIONS
--------------------------------------------------------------------------------------------------------------
Documentation             : /home/jovyan/chest-xray-ai-assistant/docs/testing_mlflow_reproducibility.md
Reusable API tests        : INCLUDED
Post-deployment validation: INCLUDED
MLflow usage              : INCLUDED
Evidence locations        : INCLUDED
Reproduction order        : INCLUDED
Retraining                : NOT REQUIRED

READY FOR STATIC PACKAGING VALIDATION


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-15-static-packaging-validation"></a>
## 15. Static Packaging Validation

This section validates YAML parsing, launch-script syntax, Python compilation, Dockerfile contracts, Compose service wiring, health checks, persistent-volume mapping, GPU reservation, pinned requirements, and documentation completeness. It does not invoke Docker builds or modify running services.


In [15]:
import json
import subprocess
from datetime import datetime, timezone

PACKAGING_VALIDATION_PATH = PACKAGING_OUTPUT_ROOT / "static_packaging_validation.json"

bash_validation_results = [
    subprocess.run(
        ["bash", "-n", str(script_path)],
        capture_output=True,
        text=True,
        check=False,
    )
    for script_path in (API_LAUNCH_PATH, UI_LAUNCH_PATH)
]

packaging_checks = {
    "combined_requirements_available": REQUIREMENTS_PATH.is_file(),
    "api_requirements_available": API_REQUIREMENTS_PATH.is_file(),
    "ui_requirements_available": UI_REQUIREMENTS_PATH.is_file(),
    "environment_template_available": ENV_EXAMPLE_PATH.is_file(),
    "runtime_contract_parses": isinstance(persisted_runtime_contract, dict),
    "launch_scripts_parse": all(
        result.returncode == 0
        for result in bash_validation_results
    ),
    "api_dockerfile_contract": all(item in api_docker_text for item in ("api.main:app", "/health", "cuda12.1")),
    "ui_dockerfile_contract": all(item in ui_docker_text for item in ("ui/app.py", "/_stcore/health", "http://api:8000")),
    "compose_services_available": set(persisted_compose["services"]) == {"api", "ui"},
    "compose_api_healthcheck": "healthcheck" in api_service,
    "compose_ui_healthcheck": "healthcheck" in ui_service,
    "compose_persistent_volume": bool(api_service.get("volumes")),
    "compose_gpu_reservation": "deploy" in api_service,
    "compose_health_gated_dependency": ui_service.get("depends_on", {}).get("api", {}).get("condition") == "service_healthy",
    "deployment_validator_compiles": DEPLOYMENT_VALIDATOR_PATH.is_file(),
    "readme_complete": not missing_readme_sections,
    "architecture_complete": ARCHITECTURE_PATH.is_file(),
    "usage_complete": USAGE_PATH.is_file(),
    "api_examples_complete": not missing_examples,
    "safety_complete": not missing_safety,
    "testing_mlflow_complete": TESTING_MLFLOW_PATH.is_file(),
}

failed_packaging_checks = [name for name, passed in packaging_checks.items() if passed is not True]
if failed_packaging_checks:
    raise RuntimeError(f"Static packaging validation failed: {failed_packaging_checks}")

packaging_validation = {
    "validation_version": "cloud-native-static-packaging-validation-v1",
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "docker_build_executed": False,
    "services_restarted": False,
    "models_loaded": False,
    "checks": packaging_checks,
}
PACKAGING_VALIDATION_PATH.write_text(
    json.dumps(packaging_validation, indent=2), encoding="utf-8"
)

print("STATIC PACKAGING VALIDATION")
print("-" * 110)
for name, passed in packaging_checks.items():
    print(f"{name:<62}: {'PASS' if passed else 'FAIL'}")
print()
print(f"Validated checks           : {len(packaging_checks)}")
print(f"Failed checks              : 0")
print(f"Evidence artifact          : {PACKAGING_VALIDATION_PATH}")
print(f"Docker build executed      : NO")
print(f"Services restarted         : NO")
print(f"Models loaded              : NO")
print()
print("READY FOR DEMONSTRATION EVIDENCE AUDIT")


STATIC PACKAGING VALIDATION
--------------------------------------------------------------------------------------------------------------
combined_requirements_available                               : PASS
api_requirements_available                                    : PASS
ui_requirements_available                                     : PASS
environment_template_available                                : PASS
runtime_contract_parses                                       : PASS
launch_scripts_parse                                          : PASS
api_dockerfile_contract                                       : PASS
ui_dockerfile_contract                                        : PASS
compose_services_available                                    : PASS
compose_api_healthcheck                                       : PASS
compose_ui_healthcheck                                        : PASS
compose_persistent_volume                                     : PASS
compose_gpu_reservation          

**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-16-demonstration-evidence-audit"></a>
## 16. Demonstration Evidence Audit

This section verifies the completed Notebook 8 live-workflow and screenshot-readiness evidence, preserves the authenticated Streamlit route and non-clinical demonstration input in documentation, and avoids generating or submitting another image.


In [16]:
SCREENSHOT_READINESS_PATH = UI_OUTPUT_ROOT / "screenshot_readiness_validation.json"
LIVE_WORKFLOW_PATH = UI_OUTPUT_ROOT / "live_interface_workflow_validation.json"
UI_DEMO_INPUT_PATH = REPORTS_ROOT / "streamlit_demo_input.png"
DEMONSTRATION_DOC_PATH = DOCS_ROOT / "demonstration_evidence.md"
DEMONSTRATION_AUDIT_PATH = PACKAGING_OUTPUT_ROOT / "demonstration_evidence_audit.json"

for evidence_path in (SCREENSHOT_READINESS_PATH, LIVE_WORKFLOW_PATH, UI_DEMO_INPUT_PATH):
    if not evidence_path.is_file():
        raise FileNotFoundError(f"Required demonstration evidence is unavailable: {evidence_path}")

screenshot_evidence = json.loads(SCREENSHOT_READINESS_PATH.read_text(encoding="utf-8"))
live_workflow_evidence = json.loads(LIVE_WORKFLOW_PATH.read_text(encoding="utf-8"))
screenshot_checks = screenshot_evidence.get("checks", {})

demonstration_checks = {
    "screenshot_checks_passed": bool(screenshot_checks) and all(value is True for value in screenshot_checks.values()),
    "streamlit_proxy_path_available": bool(screenshot_evidence.get("jupyter_proxy_path")),
    "demonstration_input_available": UI_DEMO_INPUT_PATH.stat().st_size > 0,
    "live_prediction_available": bool(live_workflow_evidence.get("prediction_id")),
    "stored_prediction_verified": live_workflow_evidence.get("stored_prediction_verified") is True,
    "operational_metrics_verified": live_workflow_evidence.get("operational_metrics_verified") is True,
    "render_exceptions_absent": live_workflow_evidence.get("streamlit_render_exceptions") == 0,
    "dataset_not_accessed": live_workflow_evidence.get("dataset_accessed") is False,
}
failed_demo_checks = [name for name, passed in demonstration_checks.items() if passed is not True]
if failed_demo_checks:
    raise RuntimeError(f"Demonstration evidence audit failed: {failed_demo_checks}")

demonstration_text = f'''
<!-- Managed by Notebook 9 -->
# Demonstration Evidence

- Streamlit authenticated route: `{screenshot_evidence['jupyter_proxy_path']}`
- Demonstration input: `{UI_DEMO_INPUT_PATH}`
- Screenshot-readiness checks: `{len(screenshot_checks)} / {len(screenshot_checks)} passed`
- Live workflow prediction: `{live_workflow_evidence['prediction_id']}`
- Findings returned: `{live_workflow_evidence.get('finding_count')}`
- Stored prediction retrieval: `PASS`
- Operational metrics retrieval: `PASS`
- Streamlit rendering exceptions: `0`
- Dataset archive accessed during validation: `NO`

Manual screenshots must retain the application title, exact educational limitation, upload and analysis workflow, cautious finding wording, Grad-CAM limitation, grounded outputs, follow-up workflow, stored-prediction retrieval, and operational metrics. Screenshots must not expose credentials, private paths, personal health information, or unsupported clinical claims.
'''
write_managed_text(DEMONSTRATION_DOC_PATH, demonstration_text, "<!-- Managed by Notebook 9 -->")

demonstration_audit = {
    "audit_version": "final-demonstration-evidence-audit-v1",
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "checks": demonstration_checks,
    "screenshot_readiness_path": str(SCREENSHOT_READINESS_PATH),
    "live_workflow_path": str(LIVE_WORKFLOW_PATH),
    "demonstration_input_path": str(UI_DEMO_INPUT_PATH),
}
DEMONSTRATION_AUDIT_PATH.write_text(
    json.dumps(demonstration_audit, indent=2), encoding="utf-8"
)

print("DEMONSTRATION EVIDENCE AUDIT")
print("-" * 100)
print(f"Screenshot checks         : {len(screenshot_checks)} / {len(screenshot_checks)} passed")
print(f"Live workflow             : PASS")
print(f"Demonstration input       : {UI_DEMO_INPUT_PATH}")
print(f"Demonstration document    : {DEMONSTRATION_DOC_PATH}")
print(f"Evidence audit            : {DEMONSTRATION_AUDIT_PATH}")
print(f"Image submitted           : NO")
print(f"Model inference triggered : NO")
print()
print("READY FOR FINAL ARTIFACT REGISTRY")


DEMONSTRATION EVIDENCE AUDIT
----------------------------------------------------------------------------------------------------
Screenshot checks         : 22 / 22 passed
Live workflow             : PASS
Demonstration input       : /home/jovyan/chest-xray-ai-assistant/reports/streamlit_demo_input.png
Demonstration document    : /home/jovyan/chest-xray-ai-assistant/docs/demonstration_evidence.md
Evidence audit            : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/packaging/demonstration_evidence_audit.json
Image submitted           : NO
Model inference triggered : NO

READY FOR FINAL ARTIFACT REGISTRY


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-17-final-artifact-registry"></a>
## 17. Final Artifact Registry

This section creates a checksum-protected final registry for packaging files, documentation, launch and validation utilities, core API/UI source, persisted configuration, and upstream API/UI readiness evidence. Large model and dataset artifacts remain represented by their established upstream registries rather than being rehashed.


In [17]:
import hashlib
import json
from datetime import datetime, timezone

FINAL_ARTIFACT_REGISTRY_PATH = PACKAGING_OUTPUT_ROOT / "final_solution_artifact_registry.json"

explicit_final_artifacts = {
    "packaging": [
        REQUIREMENTS_PATH, API_REQUIREMENTS_PATH, UI_REQUIREMENTS_PATH,
        ENV_EXAMPLE_PATH, DOCKERFILE_API_PATH, DOCKERFILE_UI_PATH,
        COMPOSE_PATH, RUNTIME_CONTRACT_PATH, API_LAUNCH_PATH,
        UI_LAUNCH_PATH, DEPLOYMENT_VALIDATOR_PATH,
    ],
    "documentation": [
        README_PATH, ARCHITECTURE_PATH, USAGE_PATH, API_EXAMPLES_PATH,
        SAFETY_PATH, TESTING_MLFLOW_PATH, DEMONSTRATION_DOC_PATH,
    ],
    "packaging_evidence": [
        REPOSITORY_AUDIT_PATH, PACKAGING_VALIDATION_PATH,
        DEMONSTRATION_AUDIT_PATH,
    ],
    "upstream_readiness": [
        API_OUTPUT_ROOT / "api_integration_readiness.json",
        API_OUTPUT_ROOT / "api_artifact_registry.json",
        NOTEBOOK8_READINESS_PATH,
        UI_OUTPUT_ROOT / "ui_artifact_registry.json",
    ],
}

for source_dir, category in ((API_ROOT, "api_source"), (UI_ROOT, "ui_source")):
    explicit_final_artifacts[category] = sorted(
        path for path in source_dir.rglob("*.py")
        if "__pycache__" not in path.parts
    )

registry_records = []
seen_paths = set()

for category, paths in explicit_final_artifacts.items():
    for path in paths:
        path = Path(path)
        if path in seen_paths:
            continue
        if not path.is_file():
            raise FileNotFoundError(f"Final registry artifact is unavailable: {path}")
        seen_paths.add(path)
        registry_records.append(
            {
                "category": category,
                "path": str(path),
                "size_bytes": path.stat().st_size,
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            }
        )

category_counts = {}
for record in registry_records:
    category_counts[record["category"]] = category_counts.get(record["category"], 0) + 1

final_artifact_registry = {
    "registry_version": "final-solution-artifact-registry-v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "solution_title": "API-Driven Chest X-Ray Analysis and Explanation Assistant using ChestMNIST",
    "educational_use_only": True,
    "large_artifact_policy": "represented_by_upstream_checksum_registries",
    "registered_artifact_files": len(registry_records),
    "category_counts": category_counts,
    "artifacts": registry_records,
}
FINAL_ARTIFACT_REGISTRY_PATH.write_text(
    json.dumps(final_artifact_registry, indent=2), encoding="utf-8"
)

persisted_final_registry = json.loads(
    FINAL_ARTIFACT_REGISTRY_PATH.read_text(encoding="utf-8")
)
invalid_registry_records = [
    record["path"] for record in persisted_final_registry["artifacts"]
    if hashlib.sha256(Path(record["path"]).read_bytes()).hexdigest() != record["sha256"]
]
if invalid_registry_records:
    raise RuntimeError(f"Final artifact checksums failed: {invalid_registry_records}")

print("FINAL ARTIFACT REGISTRY")
print("-" * 110)
print(f"Registry path              : {FINAL_ARTIFACT_REGISTRY_PATH}")
print(f"Registry version           : final-solution-artifact-registry-v1")
print(f"Registered artifact files  : {len(registry_records)}")
for category, count in sorted(category_counts.items()):
    print(f"{category:<27}: {count}")
print(f"Artifact checksums         : PASS")
print(f"Large artifacts rehashed   : NO")
print()
print("READY FOR MLFLOW PACKAGING REGISTRATION")


FINAL ARTIFACT REGISTRY
--------------------------------------------------------------------------------------------------------------
Registry path              : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/packaging/final_solution_artifact_registry.json
Registry version           : final-solution-artifact-registry-v1
Registered artifact files  : 49
api_source                 : 21
documentation              : 7
packaging                  : 11
packaging_evidence         : 3
ui_source                  : 3
upstream_readiness         : 4
Artifact checksums         : PASS
Large artifacts rehashed   : NO

READY FOR MLFLOW PACKAGING REGISTRATION


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-18-mlflow-packaging-and-documentation-registration"></a>
## 18. MLflow Packaging and Documentation Registration

This section records final packaging, documentation, validation, demonstration, and registry lineage in the persisted MLflow store. It logs only compact artifacts and metrics and does not load models or repeat evaluation.


In [18]:
import json
import mlflow
from datetime import datetime, timezone
from mlflow.tracking import MlflowClient

FINAL_MLFLOW_EXPERIMENT_NAME = "chestmnist-final-packaging-deployment"
FINAL_MLFLOW_REGISTRATION_PATH = PACKAGING_OUTPUT_ROOT / "final_packaging_mlflow_registration.json"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
final_experiment = mlflow_client.get_experiment_by_name(FINAL_MLFLOW_EXPERIMENT_NAME)
FINAL_MLFLOW_EXPERIMENT_ID = (
    final_experiment.experiment_id if final_experiment is not None
    else mlflow_client.create_experiment(FINAL_MLFLOW_EXPERIMENT_NAME)
)

with mlflow.start_run(
    experiment_id=FINAL_MLFLOW_EXPERIMENT_ID,
    run_name="notebook9-final-packaging-deployment",
    tags={
        "notebook": "09_packaging_deployment_documentation",
        "solution_stage": "final_packaging",
        "educational_use_only": "true",
        "models_loaded": "false",
        "models_retrained": "false",
    },
) as final_run:
    FINAL_MLFLOW_RUN_ID = final_run.info.run_id
    mlflow.log_params(
        {
            "runtime_contract_version": persisted_runtime_contract["contract_version"],
            "artifact_registry_version": final_artifact_registry["registry_version"],
            "api_image_base": "pytorch-2.5.1-cuda12.1-cudnn9",
            "ui_image_base": "python-3.11-slim",
            "api_application": "api.main:app",
            "ui_application": "ui/app.py",
            "compose_services": "api,ui",
            "notebook8_status": notebook8_readiness["status"],
        }
    )
    mlflow.log_metrics(
        {
            "static_packaging_checks_passed": len(packaging_checks),
            "demonstration_checks_passed": len(demonstration_checks),
            "registered_artifact_files": len(registry_records),
            "documentation_files": category_counts.get("documentation", 0),
            "free_storage_gib": free_storage_gib,
        }
    )
    for artifact_path in (
        FINAL_ARTIFACT_REGISTRY_PATH,
        PACKAGING_VALIDATION_PATH,
        DEMONSTRATION_AUDIT_PATH,
        RUNTIME_CONTRACT_PATH,
        COMPOSE_PATH,
        README_PATH,
    ):
        mlflow.log_artifact(str(artifact_path), artifact_path="final_packaging")

final_run_record = mlflow_client.get_run(FINAL_MLFLOW_RUN_ID)
if final_run_record.info.status != "FINISHED":
    raise RuntimeError("Final MLflow packaging run did not finish successfully.")

final_mlflow_registration = {
    "registration_version": "final-packaging-mlflow-registration-v1",
    "registered_at_utc": datetime.now(timezone.utc).isoformat(),
    "tracking_uri": MLFLOW_TRACKING_URI,
    "experiment_name": FINAL_MLFLOW_EXPERIMENT_NAME,
    "experiment_id": FINAL_MLFLOW_EXPERIMENT_ID,
    "run_id": FINAL_MLFLOW_RUN_ID,
    "run_status": final_run_record.info.status,
    "models_loaded": False,
    "models_retrained": False,
}
FINAL_MLFLOW_REGISTRATION_PATH.write_text(
    json.dumps(final_mlflow_registration, indent=2), encoding="utf-8"
)

print("MLFLOW PACKAGING AND DOCUMENTATION REGISTRATION")
print("-" * 110)
print(f"Tracking URI              : {MLFLOW_TRACKING_URI}")
print(f"Experiment name           : {FINAL_MLFLOW_EXPERIMENT_NAME}")
print(f"Experiment ID             : {FINAL_MLFLOW_EXPERIMENT_ID}")
print(f"Run ID                    : {FINAL_MLFLOW_RUN_ID}")
print(f"Run status                : {final_run_record.info.status}")
print(f"Registration artifact     : {FINAL_MLFLOW_REGISTRATION_PATH}")
print(f"Models loaded             : NO")
print(f"Models retrained          : NO")
print()
print("READY FOR FINAL REPRODUCIBILITY AND STORAGE AUDIT")


/opt/conda/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


MLFLOW PACKAGING AND DOCUMENTATION REGISTRATION
--------------------------------------------------------------------------------------------------------------
Tracking URI              : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
Experiment name           : chestmnist-final-packaging-deployment
Experiment ID             : 741138967096502653
Run ID                    : 9963aa56d16d4357b8e2c65f02d668b9
Run status                : FINISHED
Registration artifact     : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/packaging/final_packaging_mlflow_registration.json
Models loaded             : NO
Models retrained          : NO

READY FOR FINAL REPRODUCIBILITY AND STORAGE AUDIT


**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-19-final-reproducibility-and-storage-audit"></a>
## 19. Final Reproducibility and Storage Audit

This section verifies pinned runtime contracts, deterministic launch targets, documented reproduction order, checksum stability, upstream readiness, optional Docker Compose parsing when available, and the protected storage reserve. Missing Docker tooling is reported without invalidating the already completed static packaging contract.


In [19]:
import json
import shutil
import subprocess
from datetime import datetime, timezone

REPRODUCIBILITY_AUDIT_PATH = PACKAGING_OUTPUT_ROOT / "final_reproducibility_storage_audit.json"

docker_executable = shutil.which("docker")
docker_compose_checked = False
docker_compose_valid = None
docker_compose_details = "Docker CLI is not available; static YAML validation is authoritative."

if docker_executable:
    compose_process = subprocess.run(
        [docker_executable, "compose", "-f", str(COMPOSE_PATH), "config", "--quiet"],
        cwd=str(SOLUTION_ROOT), capture_output=True, text=True, check=False,
    )
    docker_compose_checked = True
    docker_compose_valid = compose_process.returncode == 0
    docker_compose_details = (
        "PASS" if docker_compose_valid
        else (compose_process.stderr.strip() or compose_process.stdout.strip())
    )

current_free_storage_gib = shutil.disk_usage(DATA_ROOT).free / (1024 ** 3)
registry_checksums_stable = all(
    hashlib.sha256(Path(record["path"]).read_bytes()).hexdigest() == record["sha256"]
    for record in persisted_final_registry["artifacts"]
)

reproducibility_checks = {
    "notebook8_readiness_preserved": NOTEBOOK8_READINESS_CONFIRMED,
    "python_311_contract_preserved": persisted_runtime_contract["python_version"] == "3.11",
    "cuda_runtime_pinned": "torch==2.5.1+cu121" in API_REQUIREMENTS_PATH.read_text(encoding="utf-8"),
    "protobuf_compatibility_pinned": "protobuf==4.25.3" in API_REQUIREMENTS_PATH.read_text(encoding="utf-8"),
    "api_launch_target_preserved": persisted_runtime_contract["api"]["application"] == "api.main:app",
    "ui_http_boundary_preserved": persisted_runtime_contract["ui"]["api_base_url_environment_variable"] == "CHEST_XRAY_API_BASE_URL",
    "persistent_volume_contract_preserved": bool(api_service.get("volumes")),
    "reproduction_order_documented": "## Reproduction order" in persisted_testing_doc,
    "artifact_registry_checksums_stable": registry_checksums_stable,
    "static_packaging_validation_passed": all(packaging_checks.values()),
    "demonstration_evidence_passed": all(demonstration_checks.values()),
    "mlflow_registration_finished": final_run_record.info.status == "FINISHED",
    "protected_storage_reserve_available": current_free_storage_gib >= MINIMUM_FREE_STORAGE_GIB,
}

if docker_compose_checked:
    reproducibility_checks["docker_compose_config_valid"] = docker_compose_valid

failed_reproducibility_checks = [name for name, passed in reproducibility_checks.items() if passed is not True]
if failed_reproducibility_checks:
    raise RuntimeError(
        "Final reproducibility and storage audit failed: "
        f"{failed_reproducibility_checks}"
    )

reproducibility_audit = {
    "audit_version": "final-reproducibility-storage-audit-v1",
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "docker_compose_checked": docker_compose_checked,
    "docker_compose_result": docker_compose_details,
    "free_storage_gib": round(current_free_storage_gib, 2),
    "protected_reserve_gib": MINIMUM_FREE_STORAGE_GIB,
    "checks": reproducibility_checks,
}
REPRODUCIBILITY_AUDIT_PATH.write_text(
    json.dumps(reproducibility_audit, indent=2), encoding="utf-8"
)

print("FINAL REPRODUCIBILITY AND STORAGE AUDIT")
print("-" * 110)
for name, passed in reproducibility_checks.items():
    print(f"{name:<62}: {'PASS' if passed else 'FAIL'}")
print()
print(f"Validated checks           : {len(reproducibility_checks)}")
print(f"Failed checks              : 0")
print(f"Docker Compose CLI checked : {'YES' if docker_compose_checked else 'NO — STATIC VALIDATION USED'}")
print(f"Free storage               : {current_free_storage_gib:.2f} GiB")
print(f"Protected storage reserve  : PASS")
print(f"Audit artifact             : {REPRODUCIBILITY_AUDIT_PATH}")
print()
print("READY FOR FINAL SOLUTION READINESS GATE")


FINAL REPRODUCIBILITY AND STORAGE AUDIT
--------------------------------------------------------------------------------------------------------------
notebook8_readiness_preserved                                 : PASS
python_311_contract_preserved                                 : PASS
cuda_runtime_pinned                                           : PASS
protobuf_compatibility_pinned                                 : PASS
api_launch_target_preserved                                   : PASS
ui_http_boundary_preserved                                    : PASS
persistent_volume_contract_preserved                          : PASS
reproduction_order_documented                                 : PASS
artifact_registry_checksums_stable                            : PASS
static_packaging_validation_passed                            : PASS
demonstration_evidence_passed                                 : PASS
mlflow_registration_finished                                  : PASS
protected_storage_res

**[↑ Back to notebook index](#notebook-index)**


<a id="nb09-20-final-solution-readiness-gate"></a>
## 20. Final Solution Readiness Gate

This section performs the final evidence-based gate across Notebook 8 readiness, packaging, documentation, safety, demonstration evidence, checksum registry, MLflow lineage, reproducibility, service contracts, and storage. It persists the final solution-readiness artifact without building images, restarting services, or loading models.


In [20]:
import json
from datetime import datetime, timezone

FINAL_SOLUTION_READINESS_PATH = PACKAGING_OUTPUT_ROOT / "final_solution_readiness.json"

final_solution_checks = {
    "Notebook 8 interface readiness is preserved": NOTEBOOK8_READINESS_CONFIRMED,
    "Runtime requirements are pinned": not missing_pins,
    "Environment configuration is available": ENV_EXAMPLE_PATH.is_file(),
    "Independent API launch contract is available": API_LAUNCH_PATH.is_file(),
    "Independent UI launch contract is available": UI_LAUNCH_PATH.is_file(),
    "FastAPI container contract is valid": packaging_checks["api_dockerfile_contract"],
    "Streamlit container contract is valid": packaging_checks["ui_dockerfile_contract"],
    "Compose connectivity and volume contracts are valid": all((
        packaging_checks["compose_services_available"],
        packaging_checks["compose_persistent_volume"],
        packaging_checks["compose_health_gated_dependency"],
    )),
    "Deployment health validator is available": packaging_checks["deployment_validator_compiles"],
    "Solution README is complete": packaging_checks["readme_complete"],
    "Architecture and usage documentation are complete": all((
        packaging_checks["architecture_complete"],
        packaging_checks["usage_complete"],
    )),
    "API examples preserve authoritative fields": packaging_checks["api_examples_complete"],
    "Safety and limitations are preserved": packaging_checks["safety_complete"],
    "Testing and MLflow instructions are complete": packaging_checks["testing_mlflow_complete"],
    "Demonstration evidence is preserved": all(demonstration_checks.values()),
    "Final artifact checksums are complete": registry_checksums_stable,
    "MLflow packaging registration is finished": final_run_record.info.status == "FINISHED",
    "Reproducibility audit passed": all(reproducibility_checks.values()),
    "Protected storage reserve remains available": current_free_storage_gib >= MINIMUM_FREE_STORAGE_GIB,
    "Models were not loaded or retrained": True,
}

failed_final_solution_checks = [
    name for name, passed in final_solution_checks.items() if passed is not True
]
if failed_final_solution_checks:
    raise RuntimeError(
        "Final solution readiness validation failed: "
        f"{failed_final_solution_checks}"
    )

final_solution_readiness = {
    "readiness_version": "final-solution-readiness-v1",
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "ready_for_reproducible_packaging_and_deployment",
    "solution_title": "API-Driven Chest X-Ray Analysis and Explanation Assistant using ChestMNIST",
    "educational_use_only": True,
    "notebook8_readiness": {
        "version": notebook8_readiness["readiness_version"],
        "status": notebook8_readiness["status"],
    },
    "packaging": {
        "api_dockerfile": str(DOCKERFILE_API_PATH),
        "ui_dockerfile": str(DOCKERFILE_UI_PATH),
        "compose_file": str(COMPOSE_PATH),
        "runtime_contract": str(RUNTIME_CONTRACT_PATH),
        "static_validation_checks": len(packaging_checks),
        "container_images_built": False,
    },
    "documentation": {
        "readme": str(README_PATH),
        "documentation_files": category_counts.get("documentation", 0),
    },
    "artifact_registry": {
        "path": str(FINAL_ARTIFACT_REGISTRY_PATH),
        "version": final_artifact_registry["registry_version"],
        "registered_artifact_files": len(registry_records),
    },
    "mlflow": {
        "experiment_id": FINAL_MLFLOW_EXPERIMENT_ID,
        "run_id": FINAL_MLFLOW_RUN_ID,
        "run_status": final_run_record.info.status,
    },
    "storage": {
        "free_storage_gib": round(current_free_storage_gib, 2),
        "protected_reserve_gib": MINIMUM_FREE_STORAGE_GIB,
        "reserve_available": True,
    },
    "models_loaded": False,
    "models_retrained": False,
    "checks": final_solution_checks,
}
FINAL_SOLUTION_READINESS_PATH.write_text(
    json.dumps(final_solution_readiness, indent=2), encoding="utf-8"
)

print("FINAL SOLUTION READINESS GATE")
print("-" * 110)
for name, passed in final_solution_checks.items():
    print(f"{name:<67}: {'PASS' if passed else 'FAIL'}")
print()
print(f"Validated checks             : {len(final_solution_checks)}")
print(f"Failed checks                : 0")
print(f"Static packaging checks      : {len(packaging_checks)} passed")
print(f"Demonstration checks         : {len(demonstration_checks)} passed")
print(f"Registered artifact files    : {len(registry_records)}")
print(f"MLflow run status            : FINISHED")
print(f"Container images built       : NO — EXPLICIT DEPLOYMENT ACTION")
print(f"Models loaded                : NO")
print(f"Models retrained             : NO")
print(f"Free storage                 : {current_free_storage_gib:.2f} GiB")
print(f"Protected storage reserve    : PASS")
print(f"Final readiness artifact     : {FINAL_SOLUTION_READINESS_PATH}")
print()
print("NOTEBOOK 9 COMPLETE — SOLUTION READY FOR REPRODUCIBLE PACKAGING AND DEPLOYMENT")


FINAL SOLUTION READINESS GATE
--------------------------------------------------------------------------------------------------------------
Notebook 8 interface readiness is preserved                        : PASS
Runtime requirements are pinned                                    : PASS
Environment configuration is available                             : PASS
Independent API launch contract is available                       : PASS
Independent UI launch contract is available                        : PASS
FastAPI container contract is valid                                : PASS
Streamlit container contract is valid                              : PASS
Compose connectivity and volume contracts are valid                : PASS
Deployment health validator is available                           : PASS
Solution README is complete                                        : PASS
Architecture and usage documentation are complete                  : PASS
API examples preserve authoritative fields   

**[↑ Back to notebook index](#notebook-index)**
